In [1]:
import numpy as np
from scipy.sparse import csc_array
from scipy.sparse.linalg import gmres
from copy import deepcopy

In [3]:
def DatosMat(path):
    data = [[],[],[],[]]
    with open(path) as fobj:
        for i, line in enumerate(fobj):
           # data.append([])
            if line.rsplit() != []:
                row = line.rsplit()   
                for j in range(len(row)):
                    data[j].append(float(row[j]))     
    return data

def DatosVec(path):
    data = [[],[],[]]
    with open(path) as fobj:
        for i, line in enumerate(fobj):
           # data.append([])
            if line.rsplit() != []:
                row = line.rsplit()   
                for j in range(len(row)):
                    data[j].append(float(row[j]))     
    return data

In [10]:
def GMRES(A,b,xini,m,restart,tol=1e-8):
    """
    Flexible Arnoldi Process
    Outputs V_m+1 Z_m and H_n
    inputs: m = number of iterations per restart cycle
    restart = number of restarts
    """
    n = len(b)
    Vm = np.zeros((n,m+1),dtype=complex)
    Hm = np.zeros((m+1,m),dtype=complex)
    gm = np.zeros(m+1,dtype=complex)
    eta = np.zeros(m,dtype=complex)
    Zm = np.zeros((n,m),dtype=complex)
    M = np.identity(n,dtype=complex) #Preconditioner ... for the moment I just fix it to the identity matrix 
    x = deepcopy(xini)
    k = 0
    print("Number of iterations per cycle: {0}".format(m))
    print("Number of restarts {0}:".format(restart))
    while k<restart:
        r0 = b - A.dot(x)
        beta = np.linalg.norm(r0)
        Vm[:,0] = r0/beta
        sn = np.zeros(m,dtype=complex) #Rotation elements
        cn = np.zeros(m,dtype=complex) 
        gm[0] = beta
        #----Arnoldi process to get column j ----#
        for j in range(m):
            Zm[:,j] = M.dot(Vm[:,j])
            w = A.dot(Zm[:,j])
            for i in range(j+1):
                Hm[i,j] = np.vdot(Vm[:,i],w)
                w -= Hm[i,j]*Vm[:,i]
            #Note that Hm.size = (m+1,m)
            Hm[j+1,j] = np.linalg.norm(w)
            if Hm[j+1,j]>0:
                Vm[:,j+1] = w/Hm[j+1,j]
            Hm[0:j+1+1,j], cn[j], sn[j] = rotation(cn,sn,Hm[0:j+1+1,j],j) #Rotations to column j of Hm
            temp = gm[j]
            gm[j+1] = -sn[j]*temp
            gm[j] = np.conjugate(cn[j])*gm[j]
            
        eta = solve_upper_triangular(Hm[0:m,:],gm[0:m]) #invert the upper triangular matrix
        if k % 100 == 0:
            print("Restart number {0}".format(k))
        x +=  Zm.dot(eta)    
        res = np.linalg.norm(b - A.dot(x))
        if res < tol*np.linalg.norm(b):
            return x, res
        k += 1
    return x, res 

def rotation(cn,sn,H_col,j):
    #print("Rotation for column {0}".format(j))
    #Rotation of the column elements that are <j
    for i in range(j):
        temp = np.conjugate(cn[i])*H_col[i] + np.conjugate(sn[i])*H_col[i+1]
        H_col[i+1] = -sn[i]*H_col[i] + cn[i]*H_col[i+1]        
        H_col[i] = temp

    #Rotation of the diagonal and element right below the diagonal
    den = np.sqrt(H_col[j]*np.conjugate(H_col[j]) + H_col[j+1]*np.conjugate(H_col[j+1]) )
    s, c = H_col[j+1]/den, H_col[j]/den
    H_col[j] = H_col[j]*np.conjugate(c) + np.conjugate(s)*H_col[j+1]
    H_col[j+1] = 0.0
    return H_col, c, s

def solve_upper_triangular(A,b):
    """
    Solves a linear system defined by an upper triangular matrix
    """
    n = len(b)
    x = np.zeros(n,dtype=complex)
    for i in range(n):
        i = n-1-i
        x[i] = ( b[i] - np.dot(A[i,i+1:],x[i+1:]) ) / A[i,i]
    return x

In [12]:
Nx, Nt = 20, 20
dim = Nx*Nt*2
print(dim)
D = np.zeros((dim,dim),dtype=complex)
row, col, re, im = DatosMat("AMG/build/D1.dat")
for i in range(dim*dim):
    D[ int(row[i]), int(col[i]) ] = re[i] + 1j*im[i] 
phi = np.zeros(dim,dtype=complex)
row, col, re, im = DatosMat("AMG/build/Phi1.dat")
for i in range(dim):
    phi[i] = re[i] + 1j*im[i] 

800


In [24]:
m = 500 #iterations per cycle
restarts = 1
xini = np.random.rand(dim) + 1j*0.0 #np.zeros(b.size) #np.random.rand(N) + 1j*0.0
x, res = GMRES(D,phi,xini,m,restarts,tol=1e-8)
print("my gmres implementation: {0}, res {1}".format(x[0:2],res))

#for scipy maxiter is the number of restarts while restart is the number of iterations per restar cycle...
x_scipy, exit_code= gmres(csc_array(D), phi, xini , rtol=1e-8, restart=min(m,dim),maxiter=restarts) 
residual = np.linalg.norm(phi - D @ (x_scipy)) 
print("gmres from scipy. Exit code: {0} sol: {1}, res {2}".format(exit_code,x_scipy[0:2],residual ))

Number of iterations per cycle: 500
Number of restarts 1:
Restart number 0
my gmres implementation: [-19.70389339 +6.93731943j   4.06508702-30.89975509j], res 1.0498415613284741e-12
gmres from scipy. Exit code: 0 sol: [-19.70389337 +6.9373193j    4.06508721-30.89975489j], res 2.6257550404930765e-07


In [ ]:
%%timeit
x, res = GMRES(D,phi,xini,m,restarts,tol=1e-8)

In [ ]:
%%timeit
x_scipy, exit_code= gmres(csc_array(D), phi, xini , rtol=1e-8, restart=min(m,dim),maxiter=restarts) 

# Some things for testing

In [ ]:
N=200
A = np.random.rand(N,N)  +1j*np.random.rand(N,N)#np.array([[1,0,0],[0,2,0],[0,0,3]]) #np.random.rand(N,N)  +1j*np.random.rand(N,N)
b = np.ones(N,dtype=complex)#np.random.rand(N) +1j*np.random.rand(N)#np.array([1, 4, 6]) #np.random.rand(N) +1j*np.random.rand(N)
A_sparse = csc_array(A)

In [ ]:
np.linalg.cond(A)

In [ ]:
m = 200 #iterations per cycle
restarts = 1
xini = np.random.rand(N) + 1j*0.0 #np.zeros(b.size) #np.random.rand(N) + 1j*0.0
x, res = GMRES(A,b,xini,m,restarts,tol=1e-8)
print("gmres sol with implementation of least squares: {0}, res {1}".format(x[0:2],res))

#for scipy maxiter is the number of restarts while restart is the number of iterations per restar cycle...
x_scipy, exit_code= gmres(csc_array(A), b, xini , rtol=1e-8, restart=min(m,len(A)),maxiter=restarts) 
residual = np.linalg.norm(b - A.dot(x_scipy)) 
print("gmres from scipy. Exit code: {0} sol: {1}, res {2}".format(exit_code,x_scipy[0:2],residual ))

In [ ]:
%%timeit
x, res = GMRES(A,b,xini,m,restarts,tol=1e-8)

In [ ]:
%%timeit
x_scipy, exit_code= gmres(A_sparse, b, xini , rtol=1e-8, restart=min(m,len(A)),maxiter=restarts) 

### --------- Testing rotations for least squares ----------

In [ ]:
N=10
m = 3
A = np.random.rand(N,N)  +1j*np.random.rand(N,N)#np.array([[1,0,0],[0,2,0],[0,0,3]]) #np.random.rand(N,N)  +1j*np.random.rand(N,N)
b = np.random.rand(N) +1j*np.random.rand(N)#np.array([1, 4, 6]) #np.random.rand(N) +1j*np.random.rand(N)
xini = np.random.rand(N) + 1j*0.0 #np.zeros(b.size) #np.random.rand(N) + 1j*0.0
Hm,HM = GMRES(A,b,xini,m,restart,tol)
for i in range(Hm.shape[0]):
    for j in range(Hm.shape[1]):
        print(np.round(Hm[i,j],8),end=" ")
    print("")

In [ ]:
omega1 = np.identity(m+1,dtype=complex)
omega2 = np.identity(m+1,dtype=complex)
omega3 = np.identity(m+1,dtype=complex)

#---omega1---#
j = 0
H_col = HM[0:j+1+1,j]
den = np.sqrt(H_col[j]*np.conjugate(H_col[j]) + H_col[j+1]*np.conjugate(H_col[j+1]) )
s, c = H_col[j+1]/den, H_col[j]/den
omega1[j,j] = np.conjugate(c)
omega1[j,j+1] = np.conjugate(s)
omega1[j+1,j] = -s
omega1[j+1,j+1] = c
Rm = np.dot(omega1,HM)
#---omega2---#
j = 1
H_col = Rm[0:j+1+1,j]
den = np.sqrt(H_col[j]*np.conjugate(H_col[j]) + H_col[j+1]*np.conjugate(H_col[j+1]) )
s, c = H_col[j+1]/den, H_col[j]/den
omega2[j,j] = np.conjugate(c)
omega2[j,j+1] = np.conjugate(s)
omega2[j+1,j] = -s
omega2[j+1,j+1] = c
Rm = np.dot(omega2,Rm)
#---omega3---#
j = 2
H_col = Rm[0:j+1+1,j]
den = np.sqrt(H_col[j]*np.conjugate(H_col[j]) + H_col[j+1]*np.conjugate(H_col[j+1]) )
s, c = H_col[j+1]/den, H_col[j]/den
omega3[j,j] = np.conjugate(c)
omega3[j,j+1] = np.conjugate(s)
omega3[j+1,j] = -s
omega3[j+1,j+1] = c
Rm = np.dot(omega3,Rm)

In [ ]:
for i in range(Rm.shape[0]):
    for j in range(Rm.shape[1]):
        print(np.round(Rm[i,j],4),end=" ")
    print("")

### ----- First gmres version I wrote ------

In [ ]:
def gmres_mine(A,b,xini,m,restart,tol=1e-8):
    """
    Flexible Arnoldi Process
    Outputs V_m+1 Z_m and H_n
    inputs: m = number of iterations per restart cycle
    restart = number of restarts
    """
    n = len(b)
    restart = min(restart,n)
    Vm = np.zeros((n,m+1),dtype=complex)
    Hm = np.zeros((m+1,m),dtype=complex)
    Zm = np.zeros((n,m),dtype=complex)
    M = np.identity(n,dtype=complex) #Preconditioner ... for the moment I just fix it to the identity matrix 
    x = deepcopy(xini)
    k = 0
    print("Number of iterations per cycle: {0}".format(m))
    print("Number of restarts {0}:".format(restart))
    while k<restart:
        #Arnoldi step
        r0 = b - A.dot(x)
        beta = np.linalg.norm(r0)
        Vm[:,0] = r0/beta    
        for j in range(m):
            Zm[:,j] = M.dot(Vm[:,j])
            w = A.dot(Zm[:,j])
            for i in range(j+1):
                Hm[i,j] = np.vdot(Vm[:,i],w)
                w -= Hm[i,j]*Vm[:,i]
            #Note that Hm.size = (m+1,m)
            Hm[j+1,j] = np.linalg.norm(w)
            if Hm[j+1,j]>0:
                Vm[:,j+1] = w/Hm[j+1,j]
                
        e1 = np.eye(1,m+1,0,dtype=complex).reshape(m+1)
        eta = np.linalg.lstsq(Hm, beta*e1,rcond=None)[0]
        x +=  Zm.dot(eta)    
        res = np.linalg.norm(b - A.dot(x))
        if res < tol*np.linalg.norm(b):
            return x, res
        k += 1
    return x, res